<a href="https://colab.research.google.com/github/ai7dnn/2026-1-BDA/blob/main/code/pandas_cookbook_ko.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# pandas Cookbook 한글 문서

이 노트북은 pandas 공식 사용자 가이드의 Cookbook 페이지를 바탕으로, 설명은 한글로 정리하고 코드 예시는 원문의 형태를 최대한 유지해 재구성한 자료입니다.

- 원문: pandas User Guide - Cookbook
- 목표: 자주 쓰는 패턴을 한국어 설명과 함께 빠르게 학습
- 원칙: 설명은 한글, 코드는 그대로 또는 실행 가능한 형태로 유지

출처: https://pandas.pydata.org/docs/user_guide/cookbook.html#cookbook


Cookbook 페이지는 pandas에서 자주 마주치는 작업을 짧고 실용적인 예제로 모아둔 레시피 모음입니다. 원문도 'short and sweet' 예제를 지향하며, pandas와 NumPy를 중심으로 실제 데이터 처리 패턴을 빠르게 익히도록 구성되어 있습니다.

이 노트북은 원문의 큰 흐름을 따라 다음 주제를 정리합니다.
- Idioms: 조건 처리, 분할, 조건식 만들기
- Selection: 행/열 선택, 새 열 만들기
- MultiIndexing: 계층형 인덱스 다루기
- Missing data / Grouping / Pivot / Apply
- Timeseries / Merge / IO / Computation / Timedelta
- 예제 데이터 생성과 상수 시리즈 판별


## Selection

이 절은 DataFrame에서 조건 기반 선택, loc/iloc 슬라이싱, 보조 열 생성 같은 기본 조작을 다룹니다. pandas 사용 빈도가 매우 높은 영역입니다.


## MultiIndexing

이 절은 행 또는 열에 계층형 인덱스를 부여해 다차원 구조처럼 다루는 방법을 보여줍니다. 복잡한 표 구조를 정리하거나 부분 선택할 때 매우 유용합니다.


## Missing data

결측값을 채우는 간단한 패턴을 소개합니다. 시계열에서 앞/뒤 값으로 결측을 메우는 작업은 매우 흔합니다.


## Grouping

groupby는 pandas의 핵심 기능 중 하나입니다. 그룹별 집계뿐 아니라 apply, transform, shift, resample과 결합해 복잡한 분석도 처리할 수 있습니다.


## Pivot

pivot_table은 범주형 데이터를 요약표 형태로 재구성할 때 사용합니다. 연도-월 교차표, 지역-도시 매출 요약 같은 문제에 적합합니다.


## Apply

apply는 유연하지만 남용하면 느릴 수 있습니다. 그래도 리스트 확장, 복합 계산, rolling window 커스텀 계산 같은 경우에는 유용합니다.


## 기타 주제

Cookbook에는 시계열, 병합, 파일 입출력, 상관계수 계산, timedelta 연산 등 실무형 예제가 넓게 포함되어 있습니다. 이 노트북에서는 대표 패턴 위주로 예시를 담았습니다.


---
### 시작

## Idioms

이 절은 pandas에서 자주 쓰는 '관용적 표현'을 보여줍니다. 특히 조건에 따라 값을 바꾸거나, 데이터를 나누거나, 복합 조건을 만드는 패턴이 핵심입니다.


### 기본 예제 DataFrame

이후 여러 예제에서 반복해서 사용하는 간단한 DataFrame입니다.

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame(
    {"AAA": [4, 5, 6, 7], "BBB": [10, 20, 30, 40], "CCC": [100, 50, -30, -50]}
)
df

,AAA,BBB,CCC
0,4,10,100
1,5,20,50
2,6,30,-30
3,7,40,-50


### If-then

특정 조건을 만족하는 행만 골라 값을 바꾸려면 `loc`를 사용합니다. 한 열만 바꿀 수도 있고 여러 열을 동시에 바꿀 수도 있습니다.

In [ ]:
df.loc[df.AAA >= 5]

,AAA,BBB,CCC
1,5,20,50
2,6,30,-30
3,7,40,-50


In [ ]:
df.loc[df.AAA >= 5, "BBB"] = -1
df

,AAA,BBB,CCC
0,4,10,100
1,5,-1,50
2,6,-1,-30
3,7,-1,-50


In [ ]:
df.loc[df.AAA >= 5, ["BBB", "CCC"]] = 555
df

,AAA,BBB,CCC
0,4,10,100
1,5,555,555
2,6,555,555
3,7,555,555


In [ ]:
df.loc[df.AAA < 5, ["BBB", "CCC"]] = 2000
df

,AAA,BBB,CCC
0,4,2000,2000
1,5,555,555
2,6,555,555
3,7,555,555


### df.where와 np.where

불리언 마스크를 따로 만든 뒤 `where`를 쓰면 '조건을 만족하면 유지, 아니면 다른 값 대체' 형태를 표현할 수 있습니다. `np.where`는 새 범주형 열을 만들 때 자주 사용합니다.

In [ ]:
from IPython.display import display_html
def display_side_by_side(*args):
    """여러 데이터프레임 비교가 쉽게 옆쪽으로 표시한다"""
    html_str=''
    for df in args:
        html_str += df.to_html() + '&nbsp;'*4
    display_html(html_str.replace('table','table style="display:inline"'), raw=True)

In [ ]:
df_mask = pd.DataFrame(
    {"AAA": [True] * 4, "BBB": [False] * 4, "CCC": [True, False] * 2}
)
df_mask

,AAA,BBB,CCC
0,True,False,True
1,True,False,False
2,True,False,True
3,True,False,False


In [ ]:
df = pd.DataFrame(
    {"AAA": [4, 5, 6, 7], "BBB": [10, 20, 30, 40], "CCC": [100, 50, -30, -50]}
)
display_side_by_side(df, df_mask)

,AAA,BBB,CCC
0,4,10,100
1,5,20,50
2,6,30,-30
3,7,40,-50
,AAA,BBB,CCC
0,True,False,True
1,True,False,False
2,True,False,True
3,True,False,False


In [ ]:
df.where(df_mask, -1000) # True: 원값을 유지, False: -1000으로 대체

,AAA,BBB,CCC
0,4,-1000,100
1,5,-1000,-1000
2,6,-1000,-30
3,7,-1000,-1000


In [ ]:
df

,AAA,BBB,CCC
0,4,10,100
1,5,20,50
2,6,30,-30
3,7,40,-50


In [ ]:
df = pd.DataFrame(
    {"AAA": [4, 5, 6, 7], "BBB": [10, 20, 30, 40], "CCC": [100, 50, -30, -50]}
)
df

,AAA,BBB,CCC
0,4,10,100
1,5,20,50
2,6,30,-30
3,7,40,-50


In [ ]:
df["logic"] = np.where(df["AAA"] > 5, "high", "low")
df

,AAA,BBB,CCC,logic
0,4,10,100,low
1,5,20,50,low
2,6,30,-30,high
3,7,40,-50,high


- np.where()와 df.where() 비교

| 함수       | 구조                  | 의미          |
| -------- | ------------------- | ----------- |
| np.where | (조건, True값, False값) | 완전한 if-else |
| df.where | (조건, False대체값)      | True 유지     |


### Splitting

조건식으로 DataFrame을 둘로 나누는 가장 단순한 방법입니다.

In [ ]:
df = pd.DataFrame(
    {"AAA": [4, 5, 6, 7], "BBB": [10, 20, 30, 40], "CCC": [100, 50, -30, -50]}
)
df

,AAA,BBB,CCC
0,4,10,100
1,5,20,50
2,6,30,-30
3,7,40,-50


In [ ]:
df[df.AAA <= 5]

,AAA,BBB,CCC
0,4,10,100
1,5,20,50


In [ ]:
low_df = df[df.AAA <= 5]
high_df = df[df.AAA > 5]
display_side_by_side(low_df, high_df)

,AAA,BBB,CCC
0,4,10,100
1,5,20,50
,AAA,BBB,CCC
2,6,30,-30
3,7,40,-50


### 복합 조건 만들기

`&`, `|`, `~`를 사용해 조건을 조합할 수 있습니다. 괄호를 각 조건에 명확히 씌우는 것이 중요합니다.

In [ ]:
df = pd.DataFrame(
    {"AAA": [4, 5, 6, 7], "BBB": [10, 20, 30, 40], "CCC": [100, 50, -30, -50]}
)
df

,AAA,BBB,CCC
0,4,10,100
1,5,20,50
2,6,30,-30
3,7,40,-50


In [ ]:
df.loc[(df["BBB"] < 25) & (df["CCC"] >= -40), "AAA"] # 결과는 한 열이므로 Series

,AAA
0,4
1,5


In [ ]:
df.loc[(df["BBB"] > 25) | (df["CCC"] >= 75), "AAA"] = 999
df


,AAA,BBB,CCC
0,999,10,100
1,5,20,50
2,999,30,-30
3,999,40,-50


### argsort()의 핵심 개념
“정렬된 값”이 아니라 “정렬 순서”를 반환
np.argsort(a)는 배열을 정렬하는 인덱스 배열을 반환

In [ ]:
a = np.array([10, 1, 5, 3])
a.argsort() # 정렬된 인덱스를 반환

array([1, 3, 2, 0])

In [ ]:
a[a.argsort()]

array([ 1,  3,  5, 10])

In [ ]:
df = pd.DataFrame(
    {"AAA": [4, 5, 6, 7], "BBB": [10, 20, 30, 40], "CCC": [100, 50, -30, -50]}
)
aValue = 43.0
(df.CCC - aValue).abs()

,CCC
0,57.0
1,7.0
2,73.0
3,93.0


In [ ]:
(df.CCC - aValue).abs().argsort()

,CCC
0,1
1,0
2,2
3,3


In [ ]:
df.loc[(df.CCC - aValue).abs().argsort()]

,AAA,BBB,CCC
1,5,20,50
0,4,10,100
2,6,30,-30
3,7,40,-50


### 동적으로 조건 합치기

조건이 여러 개이고 상황에 따라 개수가 바뀐다면 리스트에 모은 뒤 `functools.reduce`로 결합할 수 있습니다.

In [ ]:
df = pd.DataFrame(
    {"AAA": [4, 5, 6, 7], "BBB": [10, 20, 30, 40], "CCC": [100, 50, -30, -50]}
)
df

,AAA,BBB,CCC
0,4,10,100
1,5,20,50
2,6,30,-30
3,7,40,-50


In [ ]:
Crit1 = df.AAA <= 5.5
Crit2 = df.BBB == 10.0
Crit3 = df.CCC > -40.0

CritList = [Crit1, Crit2, Crit3]
CritList

[0     True
 1     True
 2    False
 3    False
 Name: AAA, dtype: bool,
 0     True
 1    False
 2    False
 3    False
 Name: BBB, dtype: bool,
 0     True
 1     True
 2     True
 3    False
 Name: CCC, dtype: bool]

### Python functools.reduce() 함수 설명
- reduce()는 여러 값을 하나의 결과로 “누적 축소(reduce)”하는 함수이다
- 리스트의 요소를 앞에서부터 두 개씩 결합하며 반복 연산한다
- 함수는 반드시 두 개의 인자를 받는 형태여야 한다
- 합계, 곱, 최대값, 조건 결합 등 다양한 누적 계산에 활용된다

In [ ]:
import functools

AllCrit = functools.reduce(lambda x, y: x & y, CritList)
df[AllCrit]

,AAA,BBB,CCC
0,4,10,100


In [ ]:
from functools import reduce

reduce(lambda x, y: x + y, [1, 2, 3, 4])

10

## 데이터프레임 선택

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame(
    {"AAA": [4, 5, 6, 7], "BBB": [10, 20, 30, 40], "CCC": [100, 50, -30, -50]}
)
df

,AAA,BBB,CCC
0,4,10,100
1,5,20,50
2,6,30,-30
3,7,40,-50


In [ ]:
df[(df.AAA <= 6) & (df.index.isin([0, 2, 4]))]

,AAA,BBB,CCC
0,4,10,100
2,6,30,-30


In [ ]:
df2 = df[(df.AAA <= 6) & (df.index.isin([0, 2, 4]))]
display_side_by_side(df, df2)

,AAA,BBB,CCC
0,4,10,100
1,5,20,50
2,6,30,-30
3,7,40,-50
,AAA,BBB,CCC
0,4,10,100
2,6,30,-30


### loc와 iloc

`loc`는 라벨 기준, `iloc`는 위치 기준 선택입니다. 정수 인덱스를 사용할 때 둘의 차이를 분명히 이해해야 합니다.

In [ ]:
df = pd.DataFrame(
    {"AAA": [4, 5, 6, 7], "BBB": [10, 20, 30, 40], "CCC": [100, 50, -30, -50]},
    index=["foo", "bar", "boo", "kar"],
)
df

,AAA,BBB,CCC
foo,4,10,100
bar,5,20,50
boo,6,30,-30
kar,7,40,-50


In [ ]:
df.loc["bar":"kar"]

,AAA,BBB,CCC
bar,5,20,50
boo,6,30,-30
kar,7,40,-50


In [ ]:
df[0:3]

,AAA,BBB,CCC
foo,4,10,100
bar,5,20,50
boo,6,30,-30


In [ ]:
df["bar":"kar"]

,AAA,BBB,CCC
bar,5,20,50
boo,6,30,-30
kar,7,40,-50


In [ ]:
data = {"AAA": [4, 5, 6, 7], "BBB": [10, 20, 30, 40], "CCC": [100, 50, -30, -50]}
df2 = pd.DataFrame(data=data, index=[1, 2, 3, 4])  # Note index starts at 1.
df2

,AAA,BBB,CCC
1,4,10,100
2,5,20,50
3,6,30,-30
4,7,40,-50


In [ ]:
df2.iloc[1:3]  # Position-oriented

,AAA,BBB,CCC
2,5,20,50
3,6,30,-30


In [ ]:
df2.loc[1:3]

,AAA,BBB,CCC
1,4,10,100
2,5,20,50
3,6,30,-30


In [ ]:
df2

,AAA,BBB,CCC
1,4,10,100
2,5,20,50
3,6,30,-30
4,7,40,-50


In [ ]:
df2[1:3]

,AAA,BBB,CCC
2,5,20,50
3,6,30,-30


In [ ]:
df2

,AAA,BBB,CCC
1,4,10,100
2,5,20,50
3,6,30,-30
4,7,40,-50


명시적인 슬라이싱 방법은 두 가지가 있으며, 일반적인 경우로 세 번째 방법이 있습니다.
- 위치 기반 (파이썬 슬라이싱 스타일: 끝 제외)
- 레이블 중심 (비 파이썬 슬라이싱 스타일: 끝 포함)
- 일반 (슬라이스 스타일: 슬라이스에 레이블 또는 위치 정보가 포함되어 있는지 여부에 따라 다름)

In [ ]:
iloc_result = df2.iloc[1:3]
loc_result = df2.loc[1:3]

display_side_by_side(iloc_result, loc_result)

,AAA,BBB,CCC
2,5,20,50
3,6,30,-30
,AAA,BBB,CCC
1,4,10,100
2,5,20,50
3,6,30,-30


In [ ]:
df = pd.DataFrame(
    {"AAA": [4, 5, 6, 7], "BBB": [10, 20, 30, 40], "CCC": [100, 50, -30, -50]}
)
df

,AAA,BBB,CCC
0,4,10,100
1,5,20,50
2,6,30,-30
3,7,40,-50


In [ ]:
df[~((df.AAA <= 6) & (df.index.isin([0, 2, 4])))]

,AAA,BBB,CCC
1,5,20,50
3,7,40,-50


In [ ]:
df[((df.AAA <= 6) & (df.index.isin([0, 2, 4])))]

,AAA,BBB,CCC
0,4,10,100
2,6,30,-30


### 새 열 만들기

기존 숫자 열을 범주형 설명 열로 바꾸거나, 그룹별 최소값 행을 찾는 패턴입니다.

In [ ]:
df = pd.DataFrame({"AAA": [1, 2, 1, 3], "BBB": [1, 1, 2, 2], "CCC": [2, 1, 3, 1]})
df

,AAA,BBB,CCC
0,1,1,2
1,2,1,1
2,1,2,3
3,3,2,1


In [ ]:
source_cols = df.columns
new_cols = [str(x) + "_cat" for x in source_cols]
new_cols

['AAA_cat', 'BBB_cat', 'CCC_cat']

In [ ]:
categories = {1: "Alpha", 2: "Beta", 3: "Charlie"}
print(categories.get(df.iat[0, 0]))
print(categories.get(4))

Alpha
None


In [ ]:
df[source_cols].map(categories.get)

,AAA,BBB,CCC
0,Alpha,Alpha,Beta
1,Beta,Alpha,Alpha
2,Alpha,Beta,Charlie
3,Charlie,Beta,Alpha


In [ ]:
categories = {1: "Alpha", 2: "Beta", 3: "Charlie"}

df[new_cols] = df[source_cols].map(categories.get)
df

,AAA,BBB,CCC,AAA_cat,BBB_cat,CCC_cat
0,1,1,2,Alpha,Alpha,Beta
1,2,1,1,Beta,Alpha,Alpha
2,1,2,3,Alpha,Beta,Charlie
3,3,2,1,Charlie,Beta,Alpha


In [ ]:
df = pd.DataFrame(
    {"AAA": [1, 1, 1, 2, 2, 2, 3, 3], "BBB": [2, 1, 3, 4, 5, 1, 2, 3]}
)
df

,AAA,BBB
0,1,2
1,1,1
2,1,3
3,2,4
4,2,5
5,2,1
6,3,2
7,3,3


In [ ]:
df.groupby("AAA")["BBB"]

In [ ]:
# 각 그룹에 속한 행 인덱스 목록
df.groupby("AAA")["BBB"].groups

{1: [0, 1, 2], 2: [3, 4, 5], 3: [6, 7]}

In [ ]:
df.groupby("AAA")["BBB"].get_group(1)

,BBB
0,2
1,1
2,3


In [ ]:
df.groupby("AAA")["BBB"].get_group(2)

,BBB
3,4
4,5
5,1


In [ ]:
df.groupby("AAA")["BBB"].get_group(3)

,BBB
6,2
7,3


방법 1: idxmin() 함수를 사용하여 최솟값의 인덱스를 얻습니다.

In [ ]:
df.groupby("AAA")["BBB"].idxmin()

,BBB
AAA,
1,1
2,5
3,6


In [ ]:
df.loc[df.groupby("AAA")["BBB"].idxmin()]

,AAA,BBB
1,1,1
5,2,1
6,3,2


방법 2: 정렬한 다음 각 항목에서 첫 번째 항목을 가져옵니다.

In [ ]:
df

,AAA,BBB
0,1,2
1,1,1
2,1,3
3,2,4
4,2,5
5,2,1
6,3,2
7,3,3


In [ ]:
df.sort_values(by="BBB")

,AAA,BBB
1,1,1
5,2,1
6,3,2
0,1,2
7,3,3
2,1,3
3,2,4
4,2,5


In [ ]:
df.sort_values(by="BBB").groupby("AAA", as_index=False).first()

,AAA,BBB
0,1,1
1,2,1
2,3,2


In [ ]:
df.sort_values(by="BBB").groupby("AAA", as_index=False)

In [ ]:
df.sort_values(by="BBB").groupby("AAA", as_index=False).groups

{1: [1, 0, 2], 2: [5, 3, 4], 3: [6, 7]}

### MultiIndex 기초

열 이름에 포함된 접두어를 분리해 계층형 열 인덱스로 바꾼 뒤, stack과 reset_index로 형태를 재구성할 수 있습니다.

In [ ]:
df = pd.DataFrame(
    {
        "row": [0, 1, 2],
        "One_X": [1.1, 1.1, 1.1],
        "One_Y": [1.2, 1.2, 1.2],
        "Two_X": [1.11, 1.11, 1.11],
        "Two_Y": [1.22, 1.22, 1.22],
    }
)
df

,row,One_X,One_Y,Two_X,Two_Y
0,0,1.1,1.2,1.11,1.22
1,1,1.1,1.2,1.11,1.22
2,2,1.1,1.2,1.11,1.22


In [ ]:
# 지정한 열을 행 index로 지정
df = df.set_index("row")
df

,One_X,One_Y,Two_X,Two_Y
row,,,,
0,1.1,1.2,1.11,1.22
1,1.1,1.2,1.11,1.22
2,1.1,1.2,1.11,1.22


In [ ]:
df.index

Index([0, 1, 2], dtype='int64', name='row')

In [ ]:
[tuple(c.split("_")) for c in df.columns]

[('One', 'X'), ('One', 'Y'), ('Two', 'X'), ('Two', 'Y')]

In [ ]:
df.columns = pd.MultiIndex.from_tuples([tuple(c.split("_")) for c in df.columns])
df.columns

MultiIndex([('One', 'X'),
            ('One', 'Y'),
            ('Two', 'X'),
            ('Two', 'Y')],
           )

In [ ]:
df

One        Two      
       X    Y     X     Y
row                      
0    1.1  1.2  1.11  1.22
1    1.1  1.2  1.11  1.22
2    1.1  1.2  1.11  1.22

### df.stack(0)의 동작 방식
- 원래 구조:
    - 행 인덱스 (index)
    - 열 (columns, possibly MultiIndex)
- 변환 후 구조:
    - 선택된 열 레벨이 행 인덱스로 이동
    - 결과적으로 행 인덱스가 증가하고 열 수는 감소

In [ ]:
display_side_by_side(df, df.stack(0))

/tmp/ipykernel_61277/3764189130.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  display_side_by_side(df, df.stack(0))


One 
 Two 
 
 
 
 X 
 Y 
 X 
 Y 
 
 
 row 
 
 
 
 
 
 
 
 
 0 
 1.1 
 1.2 
 1.11 
 1.22 
 
 
 1 
 1.1 
 1.2 
 1.11 
 1.22 
 
 
 2 
 1.1 
 1.2 
 1.11 
 1.22 
 
 
      
 
 
 
 
 X 
 Y 
 
 
 row 
 
 
 
 
 
 
 
 0 
 One 
 1.10 
 1.20 
 
 
 Two 
 1.11 
 1.22 
 
 
 1 
 One 
 1.10 
 1.20 
 
 
 Two 
 1.11 
 1.22 
 
 
 2 
 One 
 1.10 
 1.20 
 
 
 Two 
 1.11 
 1.22

In [ ]:
display_side_by_side(df, df.stack(1))

/tmp/ipykernel_61277/4121828742.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  display_side_by_side(df, df.stack(1))


One 
 Two 
 
 
 
 X 
 Y 
 X 
 Y 
 
 
 row 
 
 
 
 
 
 
 
 
 0 
 1.1 
 1.2 
 1.11 
 1.22 
 
 
 1 
 1.1 
 1.2 
 1.11 
 1.22 
 
 
 2 
 1.1 
 1.2 
 1.11 
 1.22 
 
 
      
 
 
 
 
 One 
 Two 
 
 
 row 
 
 
 
 
 
 
 
 0 
 X 
 1.1 
 1.11 
 
 
 Y 
 1.2 
 1.22 
 
 
 1 
 X 
 1.1 
 1.11 
 
 
 Y 
 1.2 
 1.22 
 
 
 2 
 X 
 1.1 
 1.11 
 
 
 Y 
 1.2 
 1.22

In [ ]:
df.index

Index([0, 1, 2], dtype='int64', name='row')

In [ ]:
df.columns

MultiIndex([('One', 'X'),
            ('One', 'Y'),
            ('Two', 'X'),
            ('Two', 'Y')],
           )

In [ ]:
df.stack(0).index

/tmp/ipykernel_61277/3155334174.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df.stack(0).index


MultiIndex([(0, 'One'),
            (0, 'Two'),
            (1, 'One'),
            (1, 'Two'),
            (2, 'One'),
            (2, 'Two')],
           names=['row', None])

In [ ]:
df.stack(1).index

/tmp/ipykernel_61277/101924094.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df.stack(1).index


MultiIndex([(0, 'X'),
            (0, 'Y'),
            (1, 'X'),
            (1, 'Y'),
            (2, 'X'),
            (2, 'Y')],
           names=['row', None])

## 단계별 데이터 변환 과정
- 1단계: df.stack(0)
    - → MultiIndex column의 level 0을 index로 이동
- 2단계: df.reset_index(1)
    - → 방금 추가된 index(level 1)을 다시 컬럼으로 이동
## 핵심 개념
- stack = "열 → 행"
- reset_index = "행 → 열 (일부 복구)"

In [ ]:
df.stack(0).reset_index(1)

/tmp/ipykernel_61277/964097248.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df.stack(0).reset_index(1)


,level_1,X,Y
row,,,
0,One,1.10,1.20
0,Two,1.11,1.22
1,One,1.10,1.20
1,Two,1.11,1.22
2,One,1.10,1.20
2,Two,1.11,1.22


In [ ]:
df = df.stack(0).reset_index(1)
df.columns = ["Sample", "All_X", "All_Y"]
df


/tmp/ipykernel_61277/3528973648.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(0).reset_index(1)


,Sample,All_X,All_Y
row,,,
0,One,1.10,1.20
0,Two,1.11,1.22
1,One,1.10,1.20
1,Two,1.11,1.22
2,One,1.10,1.20
2,Two,1.11,1.22


## 산술

In [ ]:
cols = pd.MultiIndex.from_tuples(
    [(x, y) for x in ["A", "B", "C"] for y in ["O", "I"]]
)
cols

MultiIndex([('A', 'O'),
            ('A', 'I'),
            ('B', 'O'),
            ('B', 'I'),
            ('C', 'O'),
            ('C', 'I')],
           )

In [ ]:
df = pd.DataFrame(np.random.randn(2, 6), index=["n", "m"], columns=cols)
df

A                   B                   C          
          O         I         O         I         O         I
n  0.950088 -0.151357 -0.103219  0.410599  0.144044  1.454274
m  0.761038  0.121675  0.443863  0.333674  1.494079 -0.205158

In [ ]:
df.loc['m', ('B', 'O')]/df.loc['m', ('C', 'O')]

np.float64(0.29708148699744474)

In [ ]:
df2 = df.div(df["C"], level=1)
df2

A                   B              C     
          O         I         O         I    O    I
n  6.595840 -0.104078 -0.716581  0.282339  1.0  1.0
m  0.509369 -0.593079  0.297081 -1.626424  1.0  1.0

### MultiIndex 슬라이싱

`xs`와 `loc`를 활용하면 계층형 인덱스의 특정 수준만 골라낼 수 있습니다.

In [ ]:
coords = [("AA", "one"), ("AA", "six"), ("BB", "one"), ("BB", "two"), ("BB", "six")]
index = pd.MultiIndex.from_tuples(coords)
df = pd.DataFrame([11, 22, 33, 44, 55], index, ["MyData"])
df

MyData
AA one      11
   six      22
BB one      33
   two      44
   six      55

In [ ]:
df.index

MultiIndex([('AA', 'one'),
            ('AA', 'six'),
            ('BB', 'one'),
            ('BB', 'two'),
            ('BB', 'six')],
           )

## pandas에서 xs(cross section(단면, 횡단면)의 약자)의 의미
### MultiIndex에서의 역할
- pandas에서는 MultiIndex 구조를
    - → “여러 축으로 구성된 데이터”로 본다
- xs()는
    - → 특정 레벨 값을 기준으로
    - → 데이터를 “슬라이스(단면 추출)”하는 함수이다

In [ ]:
df

MyData
AA one      11
   six      22
BB one      33
   two      44
   six      55

In [ ]:
df.xs("one", level=1)

,MyData
AA,11
BB,33


In [ ]:
df.xs("six", level=1)

,MyData
AA,22
BB,55


In [ ]:
df.xs("AA", level=0)

,MyData
one,11
six,22


In [ ]:
df.xs("AA")

,MyData
one,11
six,22


In [ ]:
xs_level0 = df.xs("BB", level=0, axis=0)
xs_level1 = df.xs("six", level=1, axis=0)
xs_level0, xs_level1

(     MyData
 one      33
 two      44
 six      55,
     MyData
 AA      22
 BB      55)

In [ ]:
df

MyData
AA one      11
   six      22
BB one      33
   two      44
   six      55

In [ ]:
display_side_by_side(xs_level0, xs_level1)

,MyData
one,33
two,44
six,55
,MyData
AA,22
BB,55


### 결측값 처리

뒤쪽 값으로 채우는 `bfill()` 예제입니다.

In [ ]:
np.random.seed(0)
df = pd.DataFrame(
    np.random.randn(6, 1),
    index=pd.date_range("2013-08-01", periods=6, freq="B"),
    columns=list("A"),
)
df.loc[df.index[3], "A"] = np.nan
df

,A
2013-08-01,1.764052
2013-08-02,0.400157
2013-08-05,0.978738
2013-08-06,NaN
2013-08-07,1.867558
2013-08-08,-0.977278


In [ ]:
df.bfill()

,A
2013-08-01,1.764052
2013-08-02,0.400157
2013-08-05,0.978738
2013-08-06,1.867558
2013-08-07,1.867558
2013-08-08,-0.977278


In [ ]:
df.ffill()

,A
2013-08-01,1.764052
2013-08-02,0.400157
2013-08-05,0.978738
2013-08-06,0.978738
2013-08-07,1.867558
2013-08-08,-0.977278


### GroupBy + apply

`agg`와 달리 `apply`는 각 그룹의 부분 DataFrame 자체를 받아서 여러 열을 함께 참조할 수 있습니다.

In [ ]:
df = pd.DataFrame(
    {
        "animal": "cat dog cat fish dog cat cat".split(),
        "size": list("SSMMMLL"),
        "weight": [8, 10, 11, 1, 20, 12, 12],
        "adult": [False] * 5 + [True] * 2,
    }
)
df

,animal,size,weight,adult
0,cat,S,8,False
1,dog,S,10,False
2,cat,M,11,False
3,fish,M,1,False
4,dog,M,20,False
5,cat,L,12,True
6,cat,L,12,True


In [ ]:
# List the size of the animals with the highest weight.
df.groupby("animal").apply(lambda subf: subf["size"][subf["weight"].idxmax()])

/tmp/ipykernel_61277/3491427077.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("animal").apply(lambda subf: subf["size"][subf["weight"].idxmax()])


,0
animal,
cat,L
dog,M
fish,M


In [ ]:
df.loc[df.groupby("animal")["weight"].idxmax(), ["animal", "size"]]

,animal,size
5,cat,L
4,dog,M
3,fish,M


In [ ]:
df.loc[df.groupby("animal")["weight"].idxmax(), ["animal", "size"]].set_index("animal")

,size
animal,
cat,L
dog,M
fish,M


In [ ]:
gb = df.groupby("animal")
gb.get_group("cat")


,animal,size,weight,adult
0,cat,S,8,False
2,cat,M,11,False
5,cat,L,12,True
6,cat,L,12,True


In [ ]:
gb.get_group("dog")

,animal,size,weight,adult
1,dog,S,10,False
4,dog,M,20,False


In [ ]:
gb.get_group("fish")

,animal,size,weight,adult
3,fish,M,1,False


### 종료